# Consulta de SQL Warehouse desde Python 

En este notebook aprenderemos cómo conectarnos a un SQL Warehouse desde Python y ejecutar consultas SQL. No utilizaremos directamente Spark.

Lo primero será instalar la libreria `databricks-sql-connector` que nos permitirá conectarnos a un SQL Warehouse desde Python. 


In [0]:
%pip install databricks-sql-connector
dbutils.library.restartPython()

El siguiente código debe ejecutarse en Databricks. Para que funcione, es necesario:
- Tener un SQL Warehouse configurado en tu workspace con las tablas ya creadas
- Tener un token de acceso válido (creado previamente)
- Cambiar "<tu-workspace>" por el nombre de tu workspace
- Cambiar "<id-del-warehouse>" por el id de tu SQL Warehouse

In [0]:
from databricks import sql

# El hostname está en la URL del workspace.
# El http_path lo encuentras en SQL Warehouses > tu warehouse > Detalles de conexión.
with sql.connect(
    server_hostname = "<tu-workspace>.cloud.databricks.com",
    http_path       = "/sql/1.0/warehouses/<id-del-warehouse>",
    access_token    = dbutils.secrets.get(catalog="workspace", schema="liga", key="token")
) as conexion:

    with conexion.cursor() as cursor:
        cursor.execute("""
            SELECT equipo_local, equipo_visitante, goles_local, goles_visitante
            FROM workspace.liga.partidos
            WHERE jornada = 1
        """)

        for fila in cursor.fetchall():
            print(f"{fila.equipo_local} {fila.goles_local} - "
                  f"{fila.goles_visitante} {fila.equipo_visitante}")

También se puede utilizar la libreria Pandas para mostrar los resultados de las consultas en un DataFrame.

In [0]:
import pandas as pd

with sql.connect(
    server_hostname = "<tu-workspace>.cloud.databricks.com",
    http_path       = "/sql/1.0/warehouses/<id-del-warehouse>",
    access_token    = dbutils.secrets.get(catalog="workspace", schema="liga", key="token")
) as conexion:

    with conexion.cursor() as cursor:
        cursor.execute("SELECT * FROM workspace.liga.partidos")
        df = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])

df.head()